# 07 — CSS structure and the Knill-Laflamme conditions

Phase 6 demo. We verify two things on Steane:
1. The CSS condition `H_X H_Z^T = 0 (mod 2)` — equivalently, every X-stabilizer commutes with every Z-stabilizer.
2. The Knill-Laflamme conditions for weight-1 Pauli errors: `P_C E_a^dagger E_b P_C = c_{ab} P_C` where `P_C` is the codespace projector.

Read alongside `notes/09-css-codes.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qec.pauli import Pauli
from qec.codes import steane

ATOL = 1e-10


## 1. CSS condition: H_X · H_Z^T = 0 (mod 2)

For Steane the X- and Z-side parity matrices are both `HAMMING_H`. The condition is `HAMMING_H · HAMMING_H^T = 0 (mod 2)`.


In [ ]:
H = steane.HAMMING_H
prod = (H @ H.T) % 2
print("HAMMING_H · HAMMING_H^T  (mod 2):")
print(prod)
assert np.array_equal(prod, np.zeros((3, 3), dtype=np.int8))
print("CSS condition satisfied.")


## 2. Build the codespace projector P_C

The Steane codespace is the joint +1 eigenspace of all 6 stabilizer generators. The projector onto it is
`P_C = (1/2^6) prod_k (I + s_k)`,
which we compute explicitly as a 128×128 complex matrix. Its rank should be 2 (one logical qubit → 2-dim codespace).


In [ ]:
dim = 2 ** 7
P_C = np.eye(dim, dtype=complex)
for s in steane.ALL_STABILIZERS:
    M = (np.eye(dim, dtype=complex) + s.matrix()) / 2
    P_C = M @ P_C

print(f"Tr(P_C) = {np.trace(P_C).real:.4f}  (codespace dimension; should be 2)")
print(f"||P_C^2 - P_C|| = {np.linalg.norm(P_C @ P_C - P_C):.2e}  (projector check)")


## 3. Knill-Laflamme on every pair of weight-1 errors

For each pair (E_a, E_b) of weight-1 Pauli errors, compute `M = P_C E_a^dag E_b P_C` and check that it is proportional to `P_C`. We extract the proportionality constant `c_{ab}` and report the maximum residual `|| M - c_{ab} P_C ||`.


In [ ]:
errs = list(steane._enumerate_paulis(1))
print(f"weight-1 Pauli error count: {len(errs)}")

C = np.zeros((len(errs), len(errs)), dtype=complex)
max_resid = 0.0
for a, Ea in enumerate(errs):
    for b, Eb in enumerate(errs):
        Edag_E = (Ea * Eb)  # Pauli; E_a^dag = E_a up to phase, and Pauli * Pauli is Pauli
        # Use matrix form for the KL check (definition uses adjoint).
        Ma = Ea.matrix(); Mb = Eb.matrix()
        M = P_C @ Ma.conj().T @ Mb @ P_C
        # If M = c_ab P_C, then c_ab = Tr(M) / Tr(P_C).
        c_ab = np.trace(M) / np.trace(P_C)
        C[a, b] = c_ab
        resid = np.linalg.norm(M - c_ab * P_C)
        if resid > max_resid:
            max_resid = resid

print(f"max ||P_C E_a^dag E_b P_C - c_ab P_C|| = {max_resid:.2e}  (should be ~0)")
assert max_resid < 1e-10
print("KL conditions satisfied.")


## 4. The c_{ab} matrix: structure of the corrector

`c_{ab}` is a 21×21 Hermitian matrix. Its diagonal entries are 1 (each error preserves the codespace's "weight"). Its off-diagonal entries are mostly 0 (errors with different syndromes don't interfere) — this is what makes Steane **non-degenerate**.

We visualise `|c_{ab}|`.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
im = ax.imshow(np.abs(C), cmap="viridis", vmin=0, vmax=1)
ax.set_title("|c_{ab}| over weight-1 error pairs (Steane)")
ax.set_xlabel("error index b")
ax.set_ylabel("error index a")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# Diagonal should be all 1 (or close to).
diag_min = np.min(np.abs(np.diag(C)))
diag_max = np.max(np.abs(np.diag(C)))
print(f"|c_aa| range: [{diag_min:.6f}, {diag_max:.6f}]  (should be 1 for non-degenerate code)")

# Count non-zero off-diagonals
off = np.abs(C - np.diag(np.diag(C)))
nz = int(np.sum(off > 1e-10))
print(f"non-zero off-diagonal entries: {nz}/{21*21 - 21}")


## What this notebook gates

- The CSS condition is satisfied for Steane (X- and Z-stabilizers all commute, equivalently `H · H^T = 0 mod 2`).
- The codespace projector has trace 2 (one logical qubit) and is idempotent.
- The Knill-Laflamme conditions hold to numerical precision for every pair of weight-1 errors.
- The `c_{ab}` matrix has unit-norm diagonal and (mostly) zero off-diagonals → Steane is non-degenerate on weight-1 errors.

This is the rigorous correctness gate for our distance-3 codes. Up to here we've only verified "every weight-1 syndrome is unique"; KL upgrades that to "a recovery operator exists, by construction".

Next: Phase 7 — fault tolerance basics (`notes/10`), and Phase 8 — surface code intro (`notes/11` + `demos/08`).
